## 26.1 BERT Architecture

BERT = **Bidirectional Encoder Representations from Transformers**. It is an **encoder-only Transformer** pretrained on large amounts of unlabeled text and then adapted to downstream NLP tasks.

---

## 1. Why BERT?

Before BERT, many NLP systems learned representations that were limited in how they could use context.

Consider:

> **"The bank was crowded because people were waiting for their loans."**

The representation of **bank** should depend on its surrounding words.

BERT learns **context-dependent representations**:

$$
\mathbf{h}_{bank}
=
f(\text{left context},\text{bank},\text{right context})
$$

So the representation of `"bank"` can differ between:

> "I deposited money in the **bank**."

and

> "We sat beside the river **bank**."

### The key idea

BERT uses the **Transformer encoder's self-attention** to allow every token to interact with other tokens in the sequence.

Unlike an autoregressive decoder that is normally constrained to attend only to previous tokens during generation, BERT's encoder representation can use **both left and right context**. The original BERT paper describes this as jointly conditioning on left and right context in all layers. [arXiv](https://arxiv.org/abs/1810.04805?utm_source=chatgpt.com)

---

# 2. BERT = Encoder-Only Transformer

This is the first thing you should be able to say in an interview:

> **BERT is an encoder-only Transformer model designed primarily to learn contextual representations of text.**

Conceptually:

```text
                 Input text
                     │
                     ▼
               WordPiece tokens
                     │
                     ▼
         Token + Position + Segment
                 embeddings
                     │
                     ▼
         ┌───────────────────────┐
         │   Transformer Encoder │
         │                       │
         │  Self-Attention       │
         │        ↓              │
         │  Feed Forward         │
         │        ↓              │
         │  LayerNorm / Residual │
         └───────────────────────┘
                     │
                    ...
                     │
                     ▼
               Contextualized
                representations
                     │
               ┌──────┴──────┐
               ▼             ▼
            [CLS]         token outputs
               │             │
               ▼             ▼
         Classification    NER / QA
```

BERT does **not** contain the Transformer decoder stack.

That means:

| Model | Architecture | Main strength |
|---|---|---|
| **BERT** | Encoder-only | Understanding / representation |
| **GPT-style models** | Decoder-only | Autoregressive generation |
| **T5-style models** | Encoder-decoder | Text-to-text transformation |

This distinction becomes extremely important later when you study modern LLM architectures.

---

# 3. What Does "Bidirectional" Mean?

This is an important terminology point.

**Bidirectional does not mean BERT reads the sentence twice, once left-to-right and once right-to-left.**

Instead, self-attention allows a token's representation to incorporate information from tokens on **both sides**.

Example:

```text
The animal didn't cross the street because it was too tired.
                                                        ↑
                                                       it
```

When representing `"it"`, BERT can attend to information before and after that token.

Conceptually:

```text
The animal didn't cross the street because it was too tired
←────────────────── context ──────────────────────────────→
                         ↑
                         it
```

This is one of BERT's major differences from traditional unidirectional language modeling.

---

# 4. BERT Input Representation

A BERT input isn't simply:

```text
tokens → embeddings
```

The original BERT input representation combines three types of embeddings:

$$
\boxed{
E_i =
E_i^{token}
+
E_i^{segment}
+
E_i^{position}
}
$$

### 1. Token embeddings

Represent the actual WordPiece token.

### 2. Segment/token-type embeddings

Identify whether a token belongs to sentence A or sentence B.

### 3. Position embeddings

Represent the position of each token in the sequence.

So:

```text
              Token embedding
                    +
             Segment embedding
                    +
             Position embedding
                    │
                    ▼
              BERT input vector
```

For modern Hugging Face BERT implementations, `token_type_ids` use `0` for the first segment and `1` for the second segment. [Hugging Face](https://huggingface.co/docs/transformers/model_doc/bert?utm_source=chatgpt.com)

---

# 5. `[CLS]` Token

`[CLS]` is a special token placed at the beginning of a BERT sequence.

Example:

```text
[CLS] I love NLP [SEP]
```

BERT produces a contextualized representation for every input position:

```text
[CLS]   I   love   NLP   [SEP]
  ↓     ↓     ↓     ↓      ↓
 h₀     h₁    h₂    h₃     h₄
```

The final hidden state corresponding to `[CLS]` is commonly used as a **sequence-level representation** for classification tasks.

For example:

```text
                         h_[CLS]
                            │
                            ▼
                         Classifier
                            │
                     ┌───────┴───────┐
                     ▼               ▼
                  Positive        Negative
```

This is why you will frequently see:

```python
outputs.last_hidden_state[:, 0, :]
```

when working directly with a BERT model.

Here:

- `:` → all examples in batch
- `0` → first token (`[CLS]`)
- `:` → all hidden dimensions

However, don't confuse this with saying "`[CLS]` is inherently a sentence embedding." Its usefulness depends on the task and how the model was trained/fine-tuned.

---

# 6. `[SEP]` Token

`[SEP]` marks the **boundary between sequences**.

### Single sentence

```text
[CLS] I love NLP [SEP]
```

### Sentence pair

```text
[CLS] I love NLP [SEP] It is fascinating [SEP]
```

The second case is important for tasks involving two pieces of text, such as:

- sentence-pair classification
- natural language inference
- question answering

Hugging Face's BERT tokenizer documentation describes `[SEP]` as the separator used when building multiple sequences and as the final special token of a sequence. [Hugging Face](https://huggingface.co/docs/transformers/main/model_doc/bert?utm_source=chatgpt.com)

---

# 7. WordPiece Tokenization

This connects directly to your **Module 25**.

BERT doesn't necessarily tokenize text into complete words.

It uses **WordPiece subword tokenization**.

Example:

```text
playing
   ↓
play + ##ing
```

Another example might look conceptually like:

```text
unbelievable
      ↓
un + ##believ + ##able
```

The exact split depends on the tokenizer vocabulary.

### Why?

A pure word-level vocabulary has problems:

```text
Vocabulary:
cat
dog
play
...

Unknown word:
electroencephalography
        ↓
       [UNK]
```

Subword tokenization provides a compromise:

```text
Word-level       → huge vocabulary + OOV problem
Character-level  → tiny vocabulary + very long sequences
Subword-level    → useful middle ground
```

This is why Module 25 was important preparation for BERT.

---

# 8. Attention Masks

This is one of the most important practical concepts.

Suppose we want to batch:

```text
"I love NLP"

"I love deep learning"
```

The sequences have different lengths.

We pad them:

```text
[CLS] I love NLP [SEP] [PAD] [PAD]

[CLS] I love deep learning [SEP]
```

The model should **not treat `[PAD]` as meaningful text**.

So we provide an attention mask:

```text
Tokens:
[CLS] I  love NLP [SEP] [PAD] [PAD]
  1   1   1   1    1     0     0
```

Meaning:

| Mask | Meaning |
|---:|---|
| `1` | real token → can be attended to |
| `0` | padding → ignore |

Hugging Face's current BERT documentation specifies `attention_mask` values of `1` for tokens that should be attended to and `0` for padding positions. [Hugging Face](https://huggingface.co/docs/transformers/v4.51.3/model_doc/bert?utm_source=chatgpt.com)

### Important distinction

Do **not** confuse:

```text
attention_mask
```

with:

```text
[MASK]
```

They are completely different concepts.

| Concept | Purpose |
|---|---|
| `attention_mask` | Tell the model which input positions are padding / valid |
| `[MASK]` | Special token used during masked language modeling |

This distinction is a **common interview mistake**.

---

# 9. Putting Everything Together

Consider:

```text
Sentence A:
"I love NLP"

Sentence B:
"It is fascinating"
```

BERT-style input:

```text
[CLS] I love NLP [SEP] It is fascinating [SEP]
```

Conceptually:

```text
Tokens
  │
  ▼
WordPiece Tokenizer
  │
  ▼
[CLS] I love NLP [SEP] It is fascinating [SEP]
  │
  ├───────────────┐
  │               │
  ▼               ▼
Token IDs      Segment IDs
  │               │
  └───────┬───────┘
          │
          + Position embeddings
          │
          ▼
      Input representations
          │
          ▼
  ┌─────────────────────────┐
  │ Transformer Encoder 1   │
  ├─────────────────────────┤
  │ Transformer Encoder 2   │
  ├─────────────────────────┤
  │           ...           │
  ├─────────────────────────┤
  │ Transformer Encoder N   │
  └─────────────────────────┘
          │
          ▼
  Contextualized representations
```

The original BERT paper uses the Transformer encoder architecture and pretrained the model from unlabeled text before adapting it to downstream tasks. [arXiv](https://arxiv.org/abs/1810.04805?utm_source=chatgpt.com)

---

# 10. BERT Base vs BERT Large

The original BERT paper introduced two main configurations:

| | BERT-Base | BERT-Large |
|---|---:|---:|
| Transformer layers | 12 | 24 |
| Hidden size | 768 | 1024 |
| Attention heads | 12 | 16 |
| Parameters | ~110M | ~340M |

These numbers are useful for understanding the original architecture, although modern BERT-family models have many different configurations.

---

# 11. BERT's Most Important Conceptual Shift

The architecture itself wasn't the only breakthrough.

The bigger idea was:

```text
Large unlabeled corpus
        │
        ▼
     Pretraining
        │
        ▼
General language representations
        │
     ┌──┼────────┐
     ▼  ▼        ▼
   NER  QA   Classification
```

Instead of training a separate NLP model from scratch for every task, we can:

1. **Pretrain** a general language representation model.
2. **Fine-tune** it for a specific downstream task.

This is the foundation of modern pretrained NLP.

BERT was specifically designed so that its pretrained representations could be adapted to many tasks with relatively small task-specific additions. [arXiv](https://arxiv.org/abs/1810.04805?utm_source=chatgpt.com)

---

# Must Know / Good to Know / Advanced

### 🔴 Must Know

- BERT is **encoder-only**.
- BERT produces **bidirectional contextual representations**.
- Transformer self-attention enables tokens to use surrounding context.
- WordPiece is BERT's subword tokenization approach.
- `[CLS]` is placed at the beginning.
- `[SEP]` separates sequences / marks sequence boundaries.
- `attention_mask = 1` → attend to this position.
- `attention_mask = 0` → ignore padding.
- BERT combines **token + segment + position embeddings**.
- `[CLS]` representation is commonly used for sequence-level classification.
- Pretraining → fine-tuning is the fundamental BERT paradigm.

### 🟡 Good to Know

- `token_type_ids`
- BERT-Base vs BERT-Large
- Maximum position configuration of the original BERT architecture
- Difference between `[MASK]` and `attention_mask`
- Why subwords are useful

### 🟢 Advanced

- Exact BERT pretraining objectives
- MLM masking strategy
- NSP limitations
- RoBERTa improvements
- DistilBERT distillation
- BERT attention analysis
- Encoder representation quality

Those will become much more important in the **next parts of Module 26**.

---

# 🔑 What You Should Remember

If I ask you in an interview:

> **"Explain BERT architecture."**

A strong answer should sound roughly like:

> **BERT is an encoder-only Transformer pretrained on large-scale unlabeled text. It uses bidirectional self-attention to create contextualized representations, meaning each token can incorporate information from both left and right context. Its inputs use WordPiece tokens together with token, position, and segment embeddings. Special tokens such as `[CLS]` and `[SEP]` structure the input, while attention masks prevent padding tokens from participating in attention. The resulting contextual representations can then be fine-tuned for downstream tasks such as classification, NER, and question answering.**

That's the **mental model** I want you to have before moving deeper into BERT. 🎯

---